# OpenSeed Vehicle Detection Demo

This notebook demonstrates how to use the OpenSeed model wrapper for vehicle detection and classification in traffic camera footage.

## Prerequisites

Before running this notebook, ensure you have:
1. Installed all requirements: `pip install -r requirements.txt`
2. Installed detectron2: `pip install 'git+https://github.com/facebookresearch/detectron2.git'`
3. GPU access (recommended) or CPU will work but slower

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2

from src.models import OpenSeedTrafficAnalyzer

## 1. Initialize the Analyzer

Create an instance of the OpenSeed traffic analyzer. The model will be downloaded automatically on first use.

In [ ]:
# Initialize analyzer with default settings
analyzer = OpenSeedTrafficAnalyzer(
    model_name="facebook/openseed-vitl",  # or "facebook/openseed-swinl"
    confidence_threshold=0.5,
    device=None  # Auto-detect GPU or CPU
)

# Print model information
print("Model Info:")
for key, value in analyzer.get_model_info().items():
    print(f"  {key}: {value}")

## 2. Detect Vehicles in a Single Image

Test the analyzer on a sample traffic image.

In [ ]:
# Option 1: Use a local image file
# image_path = '../data/sample_traffic_image.jpg'
# results = analyzer.detect_vehicles(image_path, return_visualizations=True)

# Option 2: Use a sample image from URL (for testing)
# First, download a sample traffic image
import requests
from io import BytesIO

# Example: Download a sample traffic image
# url = "https://example.com/traffic.jpg"  # Replace with actual image URL
# response = requests.get(url)
# image = Image.open(BytesIO(response.content))

# For now, create a placeholder
print("Note: Replace with actual traffic camera image for testing")
# results = analyzer.detect_vehicles(image, return_visualizations=True)

### Display Results

In [ ]:
# Uncomment when you have results
# print("\nDetection Summary:")
# print(f"Total Vehicles: {results['total_vehicles']}")
# print(f"\nCounts by Type:")
# for vehicle_type, count in results['counts'].items():
#     print(f"  {vehicle_type}: {count}")
# 
# if results['car_truck_ratio'] is not None:
#     print(f"\nCar/Truck Ratio: {results['car_truck_ratio']:.2f}")
# 
# # Display annotated image
# if 'annotated_image' in results:
#     plt.figure(figsize=(15, 10))
#     plt.imshow(cv2.cvtColor(results['annotated_image'], cv2.COLOR_BGR2RGB))
#     plt.axis('off')
#     plt.title('Vehicle Detections')
#     plt.tight_layout()
#     plt.show()

## 3. Process Video Frames

Process multiple frames from a video file and get aggregated statistics.

In [ ]:
def extract_video_frames(video_path, num_frames=30, fps=1):
    """
    Extract frames from video file.
    
    Args:
        video_path: Path to video file
        num_frames: Number of frames to extract
        fps: Extract one frame every N seconds
    
    Returns:
        List of frames as numpy arrays
    """
    cap = cv2.VideoCapture(video_path)
    video_fps = cap.get(cv2.CAP_PROP_FPS)
    frame_interval = int(video_fps * fps)
    
    frames = []
    frame_count = 0
    
    while len(frames) < num_frames:
        ret, frame = cap.read()
        if not ret:
            break
        
        if frame_count % frame_interval == 0:
            frames.append(frame)
        
        frame_count += 1
    
    cap.release()
    return frames

In [ ]:
# Example: Process video frames
# video_path = '../data/sample_traffic_video.mp4'
# frames = extract_video_frames(video_path, num_frames=30, fps=2)
# 
# print(f"Extracted {len(frames)} frames")
# 
# # Process frames and get aggregated results
# aggregated_results = analyzer.process_video_frames(frames, aggregate=True)
# 
# print("\nAggregated Statistics:")
# print(f"Total Frames Processed: {aggregated_results['total_frames']}")
# print(f"Average Vehicles per Frame: {aggregated_results['avg_vehicles_per_frame']:.2f}")
# print(f"\nTotal Counts:")
# for vehicle_type, count in aggregated_results['total_counts'].items():
#     print(f"  {vehicle_type}: {count}")
# print(f"\nAverage Counts per Frame:")
# for vehicle_type, avg in aggregated_results['avg_counts_per_frame'].items():
#     print(f"  {vehicle_type}: {avg:.2f}")
# if aggregated_results['avg_car_truck_ratio'] is not None:
#     print(f"\nAverage Car/Truck Ratio: {aggregated_results['avg_car_truck_ratio']:.2f}")

## 4. Analyze Traffic Patterns Over Time

Process frames and visualize temporal patterns.

In [ ]:
# Get per-frame results for time series analysis
# per_frame_results = analyzer.process_video_frames(frames, aggregate=False)
# 
# # Extract time series data
# timestamps = list(range(len(per_frame_results)))
# total_vehicles = [r['total_vehicles'] for r in per_frame_results]
# car_counts = [r['counts']['car'] for r in per_frame_results]
# truck_counts = [r['counts']['truck'] for r in per_frame_results]
# 
# # Plot temporal patterns
# fig, axes = plt.subplots(2, 1, figsize=(15, 8))
# 
# # Total vehicles over time
# axes[0].plot(timestamps, total_vehicles, marker='o', linewidth=2)
# axes[0].set_xlabel('Frame Number')
# axes[0].set_ylabel('Total Vehicles')
# axes[0].set_title('Total Vehicle Count Over Time')
# axes[0].grid(True, alpha=0.3)
# 
# # Vehicle type breakdown
# axes[1].plot(timestamps, car_counts, marker='o', label='Cars', linewidth=2)
# axes[1].plot(timestamps, truck_counts, marker='s', label='Trucks', linewidth=2)
# axes[1].set_xlabel('Frame Number')
# axes[1].set_ylabel('Count')
# axes[1].set_title('Vehicle Type Distribution Over Time')
# axes[1].legend()
# axes[1].grid(True, alpha=0.3)
# 
# plt.tight_layout()
# plt.show()

## 5. Save Results

Export detection results for further analysis.

In [ ]:
import pandas as pd
import json
from datetime import datetime

# # Convert results to DataFrame
# df_results = pd.DataFrame([
#     {
#         'frame_id': i,
#         'total_vehicles': r['total_vehicles'],
#         'cars': r['counts']['car'],
#         'trucks': r['counts']['truck'],
#         'buses': r['counts']['bus'],
#         'motorcycles': r['counts']['motorcycle'],
#         'car_truck_ratio': r['car_truck_ratio']
#     }
#     for i, r in enumerate(per_frame_results)
# ])
# 
# # Save to CSV
# timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
# output_path = f'../outputs/detection_results_{timestamp}.csv'
# df_results.to_csv(output_path, index=False)
# print(f"Results saved to {output_path}")
# 
# # Save aggregated stats as JSON
# json_path = f'../outputs/aggregated_stats_{timestamp}.json'
# with open(json_path, 'w') as f:
#     json.dump(aggregated_results, f, indent=2)
# print(f"Aggregated stats saved to {json_path}")

## Next Steps

1. **Camera Feed Integration**: Connect to Caltrans CWWP2 API to collect live footage
2. **Demographic Correlation**: Link traffic patterns to census demographic data
3. **Spatial Analysis**: Analyze traffic patterns across different locations
4. **Temporal Analysis**: Study peak hours, weekday/weekend patterns
5. **Predictive Modeling**: Build regression models to predict traffic based on demographics